# EDA - Motor Master

Exploratory analysis of the motor policies master file.

In [0]:
%pip install openpyxl

In [0]:
df_motor = spark.read.parquet(
    "/Workspace/Users/bridget.macdonnell1@gmail.com/hackathon-project/workings/policies_motor_master.parquet"
)
display(df_motor.limit(10))

In [0]:
# Trim to a smaller dataframe of features currently being used plus target and do EDA on that
variables_motor = [
    "period_start", "period_end", "exposure", "driver_age", "years_licensed", "ncb_years",
    "vehicle_value_group", "vehicle_age", "make_model", "engine_size_cc", "body_type", "transmission",
    "seats", "fuel_type", "driver_assist", "cover_type", "annual_mileage", "overnight_parking",
    "vehicle_use", "named_young_driver", "telematics_opt_in", "telematics_score", "voluntary_excess",
    "compulsory_excess", "channel", "payment_method", "missed_payments_12m", "new_business",
    "renewal_count", "days_quote_to_inception", "written_premium", "hail_days", "occupation",
    "marital_status", "credit_band", "prior_claims_5y", "licence_points", "branch_id",
    "marketing_segment", "region", "urban_density", "ultimate_claims",
]

df_motor_trim = df_motor.select(
    "address_id",
    "customer_id",
    "row_id",
    "policy_id",
    "earned_premium",
    *variables_motor,
).toPandas()
display(df_motor_trim.head(10))

In [0]:
import pandas as pd

df_motor_trim["Year"] = pd.to_datetime(df_motor_trim["period_start"]).dt.year
variables_motor.append("Year")

In [0]:
categorical_motor = [
    c for c in variables_motor
    if c not in ("period_start", "period_end")
    and (df_motor_trim[c].dtype in ("object", "bool") or c == "Year")
]

counts = {c: df_motor_trim[c].value_counts(dropna=False) for c in categorical_motor}

for c, vc in counts.items():
    print(f"--- {c} ---")
    print(vc)

excel_path = "/Workspace/Users/bridget.macdonnell1@gmail.com/hackathon-project/Excel files/Motor_EDA.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    for c, vc in counts.items():
        vc.rename("count").rename_axis(c).to_excel(writer, sheet_name=c)

print(f"Saved value counts to {excel_path}")